In [ ]:
!pip install qiskit qiskit-algorithms

In [ ]:
import numpy as np
import pandas as pd
from typing import Tuple, List, Dict

# Qiskit core imports
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
# Use StatevectorSampler (V2) or fallback to the reference Sampler
try:
    from qiskit.primitives import Sampler
except ImportError:
    from qiskit.primitives import StatevectorSampler as Sampler

# 1. Synthetic Market Data Generation
def generate_market_data(seed: int = 42) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    np.random.seed(seed)
    asset_names = ["A1", "A2", "A3", "A4", "A5", "A6", "A7", "A8", "A9", "A10"]
    mu = np.array([0.08, 0.07, 0.09, 0.03, 0.04, 0.06, 0.02, 0.07, 0.06, 0.01])
    vols = np.array([0.15, 0.16, 0.21, 0.05, 0.07, 0.18, 0.08, 0.14, 0.10, 0.008])
    N = len(asset_names)
    A = np.random.uniform(0.1, 0.7, size=(N, N))
    corr = (A + A.T) / 2.0
    np.fill_diagonal(corr, 1.0)
    D = np.diag(vols)
    sigma = D @ corr @ D
    return mu, sigma, asset_names

# 2. Mathematical QUBO & Ising Mapping Engine
class QuantumPortfolioMapper:
    def __init__(self, mu: np.ndarray, sigma: np.ndarray, q: float = 0.5, B: int = 5, P: float = 10.0):
        self.mu, self.sigma, self.q, self.B, self.P = mu, sigma, q, B, P
        self.N = len(mu)
        self.Q_diag, self.Q_off = self._build_qubo()
        self.ising_op, self.offset = self._qubo_to_ising()

    def _build_qubo(self) -> Tuple[np.ndarray, np.ndarray]:
        Q_diag = self.q * np.diag(self.sigma) - self.mu + self.P * (1.0 - 2.0 * self.B)
        Q_off = 2.0 * self.q * self.sigma + 2.0 * self.P
        np.fill_diagonal(Q_off, 0)
        return Q_diag, Q_off

    def _qubo_to_ising(self) -> Tuple[SparsePauliOp, float]:
        pauli_list = []
        for i in range(self.N):
            h_i = -0.5 * self.Q_diag[i] - 0.25 * np.sum(self.Q_off[i, :])
            z_str = ["I"] * self.N; z_str[i] = "Z"
            pauli_list.append(("".join(reversed(z_str)), h_i))
        for i in range(self.N):
            for j in range(i + 1, self.N):
                z_str = ["I"] * self.N; z_str[i] = "Z"; z_str[j] = "Z"
                pauli_list.append(("".join(reversed(z_str)), 0.25 * self.Q_off[i, j]))
        offset = 0.5 * np.sum(self.Q_diag) + 0.25 * np.sum(np.triu(self.Q_off, 1)) + self.P * (self.B ** 2)
        return SparsePauliOp.from_list(pauli_list), offset

    def evaluate_bitstring(self, bitstring: str) -> float:
        x = np.array([int(b) for b in bitstring])
        return self.q * (x.T @ self.sigma @ x) - self.mu.T @ x + self.P * ((np.sum(x) - self.B) ** 2)

# 3. Optimization Logic
if __name__ == "__main__":
    mu, sigma, asset_names = generate_market_data()
    mapper = QuantumPortfolioMapper(mu, sigma)

    # Use the sampler instance
    sampler_instance = Sampler()
    qaoa = QAOA(sampler=sampler_instance, optimizer=COBYLA(maxiter=100), reps=2)
    result = qaoa.compute_minimum_eigenvalue(mapper.ising_op)

    # Extract best result
    if hasattr(result, 'eigenstate') and isinstance(result.eigenstate, dict):
        best_bitstring = max(result.eigenstate, key=result.eigenstate.get)
        print(f"Optimized Portfolio Bitstring: {best_bitstring}")
    else:
        print("Optimization complete. Check result object for details.")

/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


Optimized Portfolio Bitstring: 0110010110


In [9]:
# ----------------------------------------------------
# 3. Hybrid Post-Processing (Classical Local Search)
# ----------------------------------------------------
def classical_local_search_refinement(
    best_bitstring: str, mapper: QuantumPortfolioMapper
) -> Tuple[str, float]:
    """Applies a classical 1-flip / 2-flip neighborhood local search."""
    current_x = np.array([int(b) for b in best_bitstring])
    current_cost = mapper.evaluate_bitstring("".join(map(str, current_x)))

    improved = True
    while improved:
        improved = False
        N = len(current_x)

        # Check all 2-swaps to preserve budget constraint
        for i in range(N):
            for j in range(N):
                if current_x[i] == 1 and current_x[j] == 0:
                    candidate_x = current_x.copy()
                    candidate_x[i], candidate_x[j] = 0, 1
                    cand_str = "".join(map(str, candidate_x))
                    cand_cost = mapper.evaluate_bitstring(cand_str)

                    if cand_cost < current_cost:
                        current_cost = cand_cost
                        current_x = candidate_x
                        improved = True
                        break
            if improved:
                break

    return "".join(map(str, current_x)), current_cost

# ----------------------------------------------------
# 4. Main Workflow Execution
# ----------------------------------------------------
if __name__ == "__main__":
    mu, sigma, asset_names = generate_market_data()
    mapper = QuantumPortfolioMapper(mu, sigma, q=0.5, B=5, P=10.0)

    print(f"Ising Hamiltonian Qubit Count: {mapper.ising_op.num_qubits}")
    print(f"Energy Shift Offset: {mapper.offset:.4f}")

    # Solve using QAOA
    sampler = Sampler()
    optimizer = COBYLA(maxiter=100)
    qaoa = QAOA(sampler=sampler, optimizer=optimizer, reps=2)

    qaoa_result = qaoa.compute_minimum_eigenvalue(mapper.ising_op)
    best_bitstring = max(qaoa_result.eigenstate, key=qaoa_result.eigenstate.get)
    raw_cost = mapper.evaluate_bitstring(best_bitstring)

    print(f"\n[QAOA] Measured Bitstring: {best_bitstring}")
    print(f"[QAOA] Raw QUBO Cost: {raw_cost:.4f}")

    # Classical Post-Processing Refinement
    refined_bitstring, refined_cost = classical_local_search_refinement(best_bitstring, mapper)
    print(f"[Hybrid Refinement] Best Bitstring: {refined_bitstring}")
    print(f"[Hybrid Refinement] Refined Cost: {refined_cost:.4f}")

    # Selected Portfolio Asset Printout
    selected_indices = [i for i, bit in enumerate(refined_bitstring) if bit == '1']
    print("\nSelected Asset Portfolio (B=5):")
    for idx in selected_indices:
        print(f" - {asset_names[idx]} (Return: {mu[idx]*100:.1f}%, Vol: {np.sqrt(sigma[idx,idx])*100:.1f}%)")


Ising Hamiltonian Qubit Count: 10
Energy Shift Offset: 24.8339


/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.12/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.12/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])



[QAOA] Measured Bitstring: 1001101011
[QAOA] Raw QUBO Cost: 9.8127
[Hybrid Refinement] Best Bitstring: 1110100110
[Hybrid Refinement] Refined Cost: 9.7631

Selected Asset Portfolio (B=5):
 - A1 (Return: 8.0%, Vol: 15.0%)
 - A2 (Return: 7.0%, Vol: 16.0%)
 - A3 (Return: 9.0%, Vol: 21.0%)
 - A5 (Return: 4.0%, Vol: 7.0%)
 - A8 (Return: 7.0%, Vol: 14.0%)
 - A9 (Return: 6.0%, Vol: 10.0%)
